# Aula 1 - Demonstrações: tokenização, atenção e famílias de modelos

**Grandes Modelos de Linguagem · Residência em Engenharia de Software Inteligente (FACOM/UFMS)**
Prof. Bruno Nogueira

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bmnogueira-ufms/LLM-ESIA-2026/blob/main/aula01/aula01_demo.ipynb)

Este notebook acompanha os slides da Aula 1. Ele tem três seções, uma para cada demonstração:

1. **Tokenização** - como um texto vira números, e por que o português gasta mais tokens.
2. **Atenção** - self-attention passo a passo, visualização da atenção do BERTimbau e embeddings contextuais.
3. **Famílias de modelos** - um encoder, um encoder-decoder e um decoder em ação.

> **Antes de começar:** no Colab, vá em *Ambiente de execução → Alterar o tipo de ambiente de execução* e escolha **GPU T4**. Tudo funciona em CPU também, só mais devagar.

## 0. Preparação do ambiente

In [ ]:
# transformers e torch já vêm instalados no Colab; instalamos o que falta
# !pip install -q tiktoken bertviz

In [ ]:
import time
import numpy as np
import pandas as pd
import torch
import transformers
import tiktoken

device = "cuda" if torch.cuda.is_available() else "cpu"
print("transformers", transformers.__version__, "| dispositivo:", device)

---
## 1. Tokenização

Um modelo de linguagem não lê letras nem palavras: lê **IDs de tokens**. Cada modelo tem o seu **tokenizador**, um dicionário fixo que converte texto em IDs e vice-versa.

Vamos comparar quatro tokenizadores:

| Tokenizador | Modelo | Algoritmo | Vocabulário |
|---|---|---|---|
| `o200k_base` (tiktoken) | GPT-4o | BPE em bytes | ≈ 200 mil |
| `gpt2` | GPT-2 (2019) | BPE em bytes | ≈ 50 mil |
| `neuralmind/bert-base-portuguese-cased` | BERTimbau | WordPiece | ≈ 30 mil |
| `Qwen/Qwen2.5-0.5B-Instruct` | Qwen 2.5 | BPE em bytes | ≈ 150 mil |

In [ ]:
from transformers import AutoTokenizer

enc_gpt4o = tiktoken.get_encoding("o200k_base")
tok_gpt2  = AutoTokenizer.from_pretrained("openai-community/gpt2")
tok_bert  = AutoTokenizer.from_pretrained("neuralmind/bert-base-portuguese-cased")
tok_qwen  = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")

def pedacos(texto, nome):
    """Devolve a lista de pedaços (tokens) de um texto para o tokenizador escolhido."""
    if nome == "GPT-4o":
        return [enc_gpt4o.decode([i]) for i in enc_gpt4o.encode(texto)]
    tok = {"GPT-2": tok_gpt2, "BERTimbau": tok_bert, "Qwen2.5": tok_qwen}[nome]
    ids = tok(texto, add_special_tokens=False)["input_ids"]
    if nome == "BERTimbau":
        return tok.convert_ids_to_tokens(ids)          # mostra os '##' do WordPiece
    return [tok.decode([i]) for i in ids]

def mostrar(texto):
    print(f'Texto: "{texto}"\n')
    for nome in ["GPT-2", "BERTimbau", "Qwen2.5", "GPT-4o"]:
        p = pedacos(texto, nome)
        print(f"{nome:10s} ({len(p):2d} tokens): " + " | ".join(p))

mostrar("Os modelos de linguagem conseguem resumir documentos.")

Repare:
- No **GPT-2**, palavras comuns do português são quebradas em vários pedaços (ele foi treinado quase só com inglês).
- No **BERTimbau**, o prefixo `##` indica que o pedaço continua a palavra anterior.
- O espaço normalmente faz parte do token seguinte (`' modelos'`).

In [ ]:
# Os IDs: é isto que o modelo de fato recebe
texto = "Olá, residentes!"
ids = enc_gpt4o.encode(texto)
print("IDs:   ", ids)
print("Tokens:", [enc_gpt4o.decode([i]) for i in ids])

### 1.1 O português gasta mais tokens?

Vamos contar tokens para o **mesmo conteúdo** em português e em inglês.

In [ ]:
pares = [
    ("A inteligência artificial está transformando a forma como as empresas atendem seus clientes. "
     "Os modelos de linguagem conseguem responder perguntas, resumir documentos e classificar mensagens automaticamente.",
     "Artificial intelligence is transforming the way companies serve their customers. "
     "Language models can answer questions, summarize documents and classify messages automatically."),
]

linhas = []
for pt, en in pares:
    for nome in ["GPT-2", "BERTimbau", "Qwen2.5", "GPT-4o"]:
        n_pt, n_en = len(pedacos(pt, nome)), len(pedacos(en, nome))
        linhas.append({
            "tokenizador": nome,
            "tokens PT": n_pt, "tokens EN": n_en,
            "PT / EN": round(n_pt / n_en, 2),
            "tokens por palavra (PT)": round(n_pt / len(pt.split()), 2),
            "tokens por palavra (EN)": round(n_en / len(en.split()), 2),
        })
pd.DataFrame(linhas)

### 1.2 Treinando um BPE do zero (e comparando com o WordPiece)

O mesmo algoritmo do slide: começamos com caracteres e, a cada passo, fundimos o par vizinho mais frequente. O `_` marca o fim da palavra.

In [ ]:
from collections import Counter

corpus = {"casa": 6, "casas": 3, "casinha": 2, "asa": 4, "asinha": 2}   # palavra: frequência

def contar_pares(palavras):
    pares = Counter()
    for simbolos, freq in palavras.items():
        for a, b in zip(simbolos, simbolos[1:]):
            pares[(a, b)] += freq
    return pares

def fundir(palavras, par):
    a, b = par
    novas = {}
    for simbolos, freq in palavras.items():
        saida, i = [], 0
        while i < len(simbolos):
            if i < len(simbolos) - 1 and simbolos[i] == a and simbolos[i + 1] == b:
                saida.append(a + b); i += 2
            else:
                saida.append(simbolos[i]); i += 1
        novas[tuple(saida)] = freq
    return novas

def treinar_bpe(corpus, n_fusoes):
    palavras = {tuple(p) + ("_",): f for p, f in corpus.items()}
    fusoes = []
    for passo in range(n_fusoes):
        pares = contar_pares(palavras)
        melhor, freq = pares.most_common(1)[0]
        palavras = fundir(palavras, melhor)
        fusoes.append(melhor)
        print(f"fusão {passo + 1}: {melhor[0]} + {melhor[1]} -> {''.join(melhor)}  ({freq} ocorrências)")
    return fusoes

fusoes = treinar_bpe(corpus, 5)

In [ ]:
def codificar_bpe(palavra, fusoes):
    simbolos = tuple(palavra) + ("_",)
    for par in fusoes:                       # aplica as fusões na ordem em que foram aprendidas
        simbolos = list(fundir({simbolos: 1}, par))[0]
    return list(simbolos)

for p in ["casinha", "asas", "casarão"]:
    print(p, "->", codificar_bpe(p, fusoes))

**WordPiece** escolhe o par com maior `freq(ab) / (freq(a) × freq(b))`, e não o mais frequente:

In [ ]:
palavras = {tuple(p) + ("_",): f for p, f in corpus.items()}
freq_simbolo = Counter()
for simbolos, f in palavras.items():
    for x in simbolos:
        freq_simbolo[x] += f
pares = contar_pares(palavras)
pontuacao = {par: c / (freq_simbolo[par[0]] * freq_simbolo[par[1]]) for par, c in pares.items()}

print("BPE escolheria:      ", pares.most_common(1)[0])
print("WordPiece escolheria:", max(pontuacao.items(), key=lambda kv: kv[1]))

### 1.3 Tokens especiais e template de chat

Alguns tokens não vêm do texto: são inseridos pelo tokenizador para marcar estrutura.

In [ ]:
# BERT: [CLS] no início, [SEP] no fim, [PAD] para igualar o tamanho das frases do lote
lote = tok_bert(["O gato dormiu.", "O cachorro latiu para o carteiro."], padding=True)
for ids in lote["input_ids"]:
    print(tok_bert.convert_ids_to_tokens(ids))

In [ ]:
# Modelos de chat: as mensagens viram UMA sequência de texto com tokens especiais
mensagens = [
    {"role": "system", "content": "Você é um assistente útil."},
    {"role": "user", "content": "Resuma o texto abaixo em uma frase."},
]
print(tok_qwen.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=True))

### 1.4 Quanto custa? Quanto cabe?

APIs cobram **por token** (entrada e saída com preços diferentes) e a **janela de contexto** é medida em tokens.

> ⚠️ Os preços abaixo são **hipotéticos**, apenas para o cálculo. Consulte a página de preços do provedor que você for usar.

In [ ]:
PRECO_ENTRADA_USD_POR_MILHAO = 0.50   # hipotético: substitua pelo preço real do provedor
PRECO_SAIDA_USD_POR_MILHAO   = 1.50   # hipotético
JANELA_CONTEXTO_TOKENS       = 128_000

def estimar(texto_entrada, tokens_saida=150, chamadas_por_dia=1000):
    t_in = len(enc_gpt4o.encode(texto_entrada))
    custo_dia = chamadas_por_dia * (t_in * PRECO_ENTRADA_USD_POR_MILHAO + tokens_saida * PRECO_SAIDA_USD_POR_MILHAO) / 1e6
    print(f"Tokens de entrada por chamada: {t_in}")
    print(f"Ocupação da janela de contexto: {100 * (t_in + tokens_saida) / JANELA_CONTEXTO_TOKENS:.3f}%")
    print(f"Custo estimado: US$ {custo_dia:.2f}/dia  ·  US$ {30 * custo_dia:.2f}/mês")

estimar("Classifique o chamado a seguir em uma das categorias: matrícula, financeiro, acesso a sistemas ou documentos. "
        "Chamado: Não consigo fazer login no portal desde ontem.")

### 1.5 Curiosidade: por que o LLM erra ao contar letras?

In [ ]:
print(pedacos("Quantas letras r tem a palavra morango?", "GPT-4o"))
# O modelo recebe ' mor' + 'ango' (dois IDs), e não as sete letras.

### ✍️ Experimente
Troque o texto abaixo por uma mensagem típica do **seu MVP** e compare os tokenizadores.

In [ ]:
mostrar("Cole aqui um texto do domínio do seu MVP.")

---
## 2. Atenção

### 2.1 Self-attention passo a passo (o exemplo do slide)

Usamos vetores Q, K e V já projetados, com dimensão $d_k = 2$:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

In [ ]:
np.set_printoptions(precision=2, suppress=True)
tokens = ["o", "gato", "dormiu"]

Q = np.array([[1, 0], [0, 1], [1, 1]], dtype=float)
K = np.array([[1, 0], [0, 2], [1, 1]], dtype=float)
V = np.array([[1, 0], [0, 1], [0.5, 0.5]], dtype=float)

def softmax(x):
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

scores = Q @ K.T                       # passo 2: comparar
pesos  = softmax(scores / np.sqrt(2))  # passo 3: normalizar
Z      = pesos @ V                     # passo 4: combinar

print("Scores Q·Kᵀ:\n", scores)
print("\nPesos de atenção (cada linha soma 1):")
print(pd.DataFrame(pesos, index=tokens, columns=tokens).round(2))
print("\nSaída Z:\n", Z)

### 2.2 Máscara causal

Em um decoder (GPT), cada token só pode olhar para o passado. Colocamos $-\infty$ acima da diagonal antes do softmax.

In [ ]:
mascara = np.triu(np.ones((3, 3)), k=1).astype(bool)   # True = posição futura
scores_mascarados = np.where(mascara, -np.inf, scores / np.sqrt(2))
print(pd.DataFrame(softmax(scores_mascarados), index=tokens, columns=tokens).round(2))

### 2.3 Máscara de padding: por que −∞ e não 0?

In [ ]:
scores_frase = np.array([2.0, 1.0, 0.5, 1.5, 1.5])    # as duas últimas posições são [PAD] (scores sem significado)
eh_pad = np.array([False, False, False, True, True])

print("sem máscara:      ", softmax(scores_frase).round(3))
print("PAD com score 0:  ", softmax(np.where(eh_pad, 0.0, scores_frase)).round(3), " <- o PAD ainda recebe peso!")
print("PAD com score -inf:", softmax(np.where(eh_pad, -np.inf, scores_frase)).round(3))

### 2.4 Codificação posicional senoidal

$PE_{(pos,2i)} = \sin(pos/10000^{2i/d})$ e $PE_{(pos,2i+1)} = \cos(pos/10000^{2i/d})$

In [ ]:
import matplotlib.pyplot as plt

def codificacao_posicional(n_posicoes, d_model):
    pos = np.arange(n_posicoes)[:, None]
    i = np.arange(d_model // 2)[None, :]
    angulos = pos / np.power(10000, 2 * i / d_model)
    pe = np.zeros((n_posicoes, d_model))
    pe[:, 0::2] = np.sin(angulos)
    pe[:, 1::2] = np.cos(angulos)
    return pe

pe = codificacao_posicional(100, 128)
plt.figure(figsize=(8, 4))
plt.imshow(pe, aspect="auto", cmap="RdBu")
plt.xlabel("dimensão"); plt.ylabel("posição"); plt.colorbar(); plt.title("Codificação posicional senoidal")
plt.show()

# posições próximas têm vetores parecidos; distantes, nem tanto
cos = lambda a, b: a @ b / (np.linalg.norm(a) * np.linalg.norm(b))
print("cos(pos 10, pos 11) =", round(cos(pe[10], pe[11]), 3))
print("cos(pos 10, pos 60) =", round(cos(pe[10], pe[60]), 3))

### 2.5 Vendo a atenção do BERTimbau com o BertViz

Use os seletores para trocar de **camada** e de **cabeça**. Procure uma cabeça em que o token `ele` olha para `animal`.

> Usamos `attn_implementation="eager"` para que o modelo devolva os pesos de atenção.

In [ ]:
from transformers import AutoModel
from bertviz import head_view

bert = AutoModel.from_pretrained("neuralmind/bert-base-portuguese-cased",
                                 output_attentions=True, attn_implementation="eager")
frase = "O animal não atravessou a rua porque ele estava cansado."
entrada = tok_bert(frase, return_tensors="pt")
with torch.no_grad():
    saida = bert(**entrada)

print("camadas:", len(saida.attentions), "| formato de cada uma:", tuple(saida.attentions[0].shape))
head_view(saida.attentions, tok_bert.convert_ids_to_tokens(entrada["input_ids"][0]))

### 2.6 Embeddings contextuais: os vários “bancos”

A saída de cada camada do BERT é um vetor **por token**, que depende das palavras vizinhas. Vamos comparar o vetor de “banco” em frases diferentes usando a **similaridade de cosseno** (1 = mesma direção).

In [ ]:
import torch.nn.functional as F

def vetor_da_palavra(frase, palavra, modelo=bert, tok=tok_bert):
    """Média dos vetores (última camada) dos subtokens que formam `palavra` na frase."""
    enc = tok(frase, return_tensors="pt")
    with torch.no_grad():
        h = modelo(**enc).last_hidden_state[0]
    palavras = frase.replace(".", " .").replace(",", " ,").split()
    alvo = palavras.index(palavra)
    pos = [i for i, w in enumerate(enc.word_ids()) if w == alvo]
    return h[pos].mean(dim=0)

frases = {
    "praça":  "Sentei no banco da praça .",
    "conta":  "Abri uma conta no banco .",
    "saque":  "Fui ao banco sacar dinheiro .",
    "dados":  "O banco de dados caiu .",
}
vetores = {k: vetor_da_palavra(f, "banco") for k, f in frases.items()}
nomes = list(vetores)
sim = [[F.cosine_similarity(vetores[a], vetores[b], dim=0).item() for b in nomes] for a in nomes]
pd.DataFrame(sim, index=nomes, columns=nomes).round(2)

**O que esperar:** “conta” e “saque” (sentido financeiro) devem ser mais parecidos entre si do que com “praça”. Em um embedding **estático**, todas as células valeriam 1.

---
## 3. As três famílias em ação

### 3.1 Encoder-only: BERTimbau completando a lacuna (MLM)

In [ ]:
from transformers import AutoModelForMaskedLM

mlm = AutoModelForMaskedLM.from_pretrained("neuralmind/bert-base-portuguese-cased")

def completar(frase, k=5):
    enc = tok_bert(frase, return_tensors="pt")
    with torch.no_grad():
        logits = mlm(**enc).logits[0]
    pos = (enc["input_ids"][0] == tok_bert.mask_token_id).nonzero()[0].item()
    probs = logits[pos].softmax(-1)
    top = probs.topk(k)
    for p, i in zip(top.values, top.indices):
        print(f"{tok_bert.decode([int(i)]):>12s}  {p.item():.1%}")

completar("O cachorro [MASK] para o carteiro.")

In [ ]:
completar("A capital do Brasil é [MASK].")

### 3.2 Encoder-decoder: um T5 traduzindo do inglês para o português

Modelo: `unicamp-dl/translation-en-pt-t5` (T5 ajustado para tradução EN→PT). O T5 recebe a tarefa como um **prefixo** no texto.

In [ ]:
from transformers import AutoModelForSeq2SeqLM

tok_t5 = AutoTokenizer.from_pretrained("unicamp-dl/translation-en-pt-t5")
t5 = AutoModelForSeq2SeqLM.from_pretrained("unicamp-dl/translation-en-pt-t5").to(device)

def traduzir(texto_en):
    enc = tok_t5("translate English to Portuguese: " + texto_en, return_tensors="pt").to(device)
    with torch.no_grad():
        out = t5.generate(**enc, max_new_tokens=80)
    return tok_t5.decode(out[0], skip_special_tokens=True)

print(traduzir("Attention is all you need."))
print(traduzir("Large language models predict the next token."))

### 3.3 Decoder-only: o próximo token, de perto

Modelo: `Qwen/Qwen2.5-0.5B-Instruct` - um LLM pequeno (meio bilhão de parâmetros) que roda até em CPU.

In [ ]:
from transformers import AutoModelForCausalLM

qwen = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct").to(device)

def proximos(texto, k=5):
    enc = tok_qwen(texto, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = qwen(**enc).logits[0, -1]
    probs = logits.softmax(-1)
    top = probs.topk(k)
    for p, i in zip(top.values, top.indices):
        print(f"{repr(tok_qwen.decode([int(i)])):>14s}  {p.item():.1%}")

proximos("A capital de Mato Grosso do Sul é")

In [ ]:
# Geração token a token (greedy): o modelo escolhe sempre o mais provável e anexa à entrada
texto = "A capital de Mato Grosso do Sul é"
ids = tok_qwen(texto, return_tensors="pt")["input_ids"].to(device)
for passo in range(8):
    with torch.no_grad():
        prox = qwen(ids).logits[0, -1].argmax()
    ids = torch.cat([ids, prox.view(1, 1)], dim=1)
    print(f"passo {passo + 1}: {tok_qwen.decode(ids[0])}")

In [ ]:
# Agora como assistente: usando o template de chat
mensagens = [
    {"role": "system", "content": "Você é um assistente que responde em português, de forma breve."},
    {"role": "user", "content": "Explique em duas frases o que é um token."},
]
enc = tok_qwen.apply_chat_template(mensagens, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(device)
inicio = time.time()
with torch.no_grad():
    out = qwen.generate(**enc, max_new_tokens=120, do_sample=False)
resposta = tok_qwen.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)
print(resposta)
print(f"\n({out.shape[1] - enc['input_ids'].shape[1]} tokens gerados em {time.time() - inicio:.1f}s)")

### 3.4 Temperatura e top-p

A temperatura divide os *logits* antes do softmax: abaixo de 1 concentra a distribuição, acima de 1 achata.

In [ ]:
enc = tok_qwen("A capital de Mato Grosso do Sul é", return_tensors="pt").to(device)
with torch.no_grad():
    logits = qwen(**enc).logits[0, -1]

for T in [0.5, 1.0, 2.0]:
    probs = (logits / T).softmax(-1)
    top = probs.topk(5)
    print(f"T = {T}: " + "  ".join(f"{tok_qwen.decode([int(i)])!r} {p.item():.0%}" for p, i in zip(top.values, top.indices)))

In [ ]:
# Amostragem: rode algumas vezes e compare as respostas
mensagens = [{"role": "user", "content": "Sugira um nome curto para um chatbot de atendimento de uma universidade."}]
enc = tok_qwen.apply_chat_template(mensagens, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(device)
for T in [0.2, 1.2]:
    for _ in range(3):
        with torch.no_grad():
            out = qwen.generate(**enc, max_new_tokens=20, do_sample=True, temperature=T, top_p=0.9)
        print(f"T = {T}:", tok_qwen.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip())

---
## Para pensar (e levar para a ficha do caso de uso)

1. Quantos tokens tem uma entrada típica do seu MVP? E a saída?
2. A tarefa do seu MVP é mais parecida com qual das três demonstrações da Seção 3?
3. O modelo de 0,5 bilhão de parâmetros acertou a capital? O que isso sugere sobre o tamanho do modelo necessário para o seu caso?